# MULTILINGUAL MCQ - Difficulty Assignment

Assigns `difficulty` (Easy / Medium / Hard) to 200 sampled rows from any of the
MCQ files in `MULTILINGUAL/` - set `DATASET` in Cell 3.

| DATASET | Rows | Options | Gold form |
|---|---|---|---|
| `indic_arc` | 5500 | 4 | letter `A`-`D` |
| `mmlu_indic` | 5500 | 4 | letter `A`-`D` |
| `boolq_indic` | 2000 | 2 | literal `yes` / `no` |
| `trivia_qa` | 1100 | 4 | letter `A`-`D` |
| `indo_aryan_lid` | 500 | 5 | literal language name |

**One notebook, not six.** The pipeline is identical for all of them: present
the options, let the model pick one, compare against gold, sum three votes. Only
two things vary - the instruction wording, and whether `answer` holds a letter
or the option text itself. Both are handled by the registry in Cell 3, so the
scoring code stays in one place instead of drifting across six copies.

**Answering is structurally constrained.** The model never generates free text.
Each option is labelled `A`, `B`, `C`... and the choice is the argmax over those
letters' token ids in a **single forward pass**. It cannot answer off-list,
there is nothing to parse, and 200 rows take about a minute per model.

**No threshold.** Unlike the translation notebooks, an MCQ answer is right or
wrong, so each model contributes 1 or 0 directly:

| Models correct | Difficulty |
|---|---|
| 3 / 3 | Easy |
| 2 / 3 | Medium |
| 0-1 / 3 | Hard |

**Output.** `<dataset>_difficulty.jsonl` - all 14 schema fields with
`difficulty` filled in, plus an audit file with each model's pick.

*Note: `trivia_qa_indic.jsonl` is byte-identical to `trivia_qa.jsonl` - same
ids, same content. Only one is in the registry; running both would score the
same rows twice.*

### Cell 1 - Install dependencies and authenticate

**An HF token is required.** Llama-3.1 and Gemma-2 are gated; Mistral is not.
Accept both licences on huggingface.co, create a **read** token, then add it in
Colab via the **key icon** as a secret named `HF_TOKEN`. Use the secret rather
than pasting the token into a cell.

In [ ]:
!pip -q install -U transformers accelerate bitsandbytes huggingface_hub

HF_OK = False
try:
    from google.colab import userdata
    from huggingface_hub import login
    login(token=userdata.get("HF_TOKEN"))
    HF_OK = True
    print("HF login OK")
except Exception as e:
    print("No HF token ({}: {})".format(type(e).__name__, e))
    print("Mistral will still work; gated Llama/Gemma will fail with a 401.")

### Cell 2 - Mount Drive

Drive is the weight cache: each model is downloaded once, quantised to 4-bit and
saved here, so later runs skip the download. Three models, about **16.5 GB**. If
you have run any of the other difficulty notebooks, they are already cached and
this one downloads nothing.

In [ ]:
import os

DRIVE_OK    = False
DRIVE_MOUNT = "/drive"
CACHE_DIR   = os.path.join(DRIVE_MOUNT, "MyDrive", "models")

try:
    from google.colab import drive
    drive.mount(DRIVE_MOUNT, force_remount=True)
    DRIVE_OK = os.path.isdir(os.path.join(DRIVE_MOUNT, "MyDrive"))
except ImportError:
    print("Not running on Colab - Drive caching disabled.")
except Exception as e:
    print("DRIVE MOUNT FAILED: {}".format(e))

if DRIVE_OK:
    os.makedirs(CACHE_DIR, exist_ok=True)
    print("Drive mounted | weight cache: {}".format(CACHE_DIR))
else:
    print("\nWARNING: no Drive - weights will NOT be cached between sessions.")

### Cell 3 - Configuration and the dataset registry

`DATASETS` is where the per-file differences live. Each entry carries only what
actually varies:

- `file` - the input filename.
- `task` - one sentence describing the task, dropped into the prompt. This is
  the real reason a single notebook works: ARC needs "answer this science
  question", LID needs "identify which language this is written in", and
  everything else about the pipeline is the same.
- `note` - printed at load time, for anything worth knowing before you run.

Output paths derive from `DATASET`, so runs never collide.

In [ ]:
import gc
import json
import random
import shutil
from collections import Counter

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

# ---- the dataset registry: only the genuinely per-file bits ----
DATASETS = {
    "indic_arc": {
        "file": "indic_arc.jsonl",
        "task": ("Answer the multiple-choice science question. The question and "
                 "options are written in an Indian language."),
        "note": "grade-school science reasoning, 11 Indian languages",
    },
    "mmlu_indic": {
        "file": "mmlu_indic.jsonl",
        "task": ("Answer the multiple-choice question. It may come from any "
                 "academic subject, and is written in an Indian language."),
        "note": "MMLU translated into 11 Indian languages",
    },
    "boolq_indic": {
        "file": "boolq_indic.jsonl",
        "task": ("Read the passage and answer the yes/no question that follows. "
                 "Answer only from the passage, not from outside knowledge."),
        "note": "2 options (yes/no); gold is the literal word, not a letter",
    },
    "trivia_qa": {
        "file": "trivia_qa.jsonl",
        "task": ("Answer the multiple-choice general-knowledge question. It is "
                 "written in an Indian language."),
        "note": "trivia_qa_indic.jsonl is an identical copy - do not run both",
    },
    "indo_aryan_lid": {
        "file": "indo_aryan_lid.jsonl",
        "task": ("Identify which Indo-Aryan language the sentence is written "
                 "in. All five options use the Devanagari script, so decide "
                 "from vocabulary and grammar, not from the script."),
        "note": ("5 options, gold is a language name. ~5% of rows mention their "
                 "own answer label in the text - a mild leak"),
    },
}

DATASET = "indic_arc"
assert DATASET in DATASETS, "DATASET must be one of {}".format(list(DATASETS))
CFG = DATASETS[DATASET]

# ---- paths (derived, so runs never collide) ----
INPUT_FILE  = CFG["file"]
OUTPUT_FILE = "{}_difficulty.jsonl".format(DATASET)
AUDIT_FILE  = "{}_audit.jsonl".format(DATASET)
PROG_DIR    = "judge_progress_{}".format(DATASET)

# ---- sampling ----
N_ROWS = 200
SEED   = 42

# ---- batching ----
BATCH_SIZE = 25

# ---- schema ----
SET_EVAL_METRIC = None            # None keeps the file's own "accuracy"

# ---- the three judges ----
USE_INSTRUCT = True
REPOS = {
    True: {"mistral": "mistralai/Mistral-7B-Instruct-v0.3",
           "llama":   "meta-llama/Llama-3.1-8B-Instruct",
           "gemma":   "google/gemma-2-9b-it"},
    False: {"mistral": "mistralai/Mistral-7B-v0.3",
            "llama":   "meta-llama/Llama-3.1-8B",
            "gemma":   "google/gemma-2-9b"},
}[USE_INSTRUCT]

MODELS = [
    {"name": "mistral", "repo": REPOS["mistral"]},
    {"name": "llama",   "repo": REPOS["llama"]},
    {"name": "gemma",   "repo": REPOS["gemma"], "attn": "eager"},
]

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,     # T4 has no bf16
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
)

SCHEMA_KEYS = [
    "id", "source", "category", "subcategory", "region", "language",
    "difficulty", "task_type", "question", "options", "answer",
    "explanation", "cultural_attr", "eval_metric",
]

os.makedirs(PROG_DIR, exist_ok=True)
print("dataset: {} -> {}".format(DATASET, INPUT_FILE))
print("  note : {}".format(CFG["note"]))
print("\nJudges ({}):".format("instruct" if USE_INSTRUCT else "base"))
for m in MODELS:
    cached = DRIVE_OK and os.path.isfile(
        os.path.join(CACHE_DIR, m["name"] + "_4bit", "config.json"))
    print("  {:<8} {:<42} {}".format(
        m["name"], m["repo"], "cached" if cached else "will download"))
print("\ndevice:", "cuda" if torch.cuda.is_available() else "CPU (will be very slow)")

### Cell 4 - Load, resolve gold, and sample

The one structural difference between these files is how `answer` encodes the
correct option:

- `indic_arc`, `mmlu_indic`, `trivia_qa` store a **letter** (`"B"`).
- `boolq_indic` stores the **literal option text** (`"yes"`).
- `indo_aryan_lid` stores a **language name** (`"Braj Bhasha"`).

`gold_index()` resolves all three to a position in the `options` list, so
everything downstream is uniform. A row whose answer matches no option is
dropped and reported rather than silently scored as wrong - that would look
like difficulty when it is a data problem.

The printed answer-position distribution is worth a glance: if the gold is
concentrated on one position, a model with a positional bias will score well
without reading the question.

In [ ]:
LETTERS_ALL = [chr(ord("A") + i) for i in range(26)]

with open(INPUT_FILE, encoding="utf-8") as f:
    all_rows = [json.loads(line) for line in f]
print("Loaded {} rows from {}".format(len(all_rows), INPUT_FILE))


def gold_index(row):
    """Position of the correct option, whatever form `answer` takes."""
    opts = row.get("options")
    if not isinstance(opts, list) or len(opts) < 2:
        return None
    ans = str(row.get("answer") or "").strip()
    if not ans:
        return None

    # form 1: a single letter within range
    if len(ans) == 1 and ans.upper() in LETTERS_ALL[:len(opts)]:
        return LETTERS_ALL.index(ans.upper())

    # form 2: the option text itself
    stripped = [str(o).strip() for o in opts]
    if ans in stripped:
        return stripped.index(ans)
    low = [s.lower() for s in stripped]
    if ans.lower() in low:
        return low.index(ans.lower())
    return None


usable, unresolved = [], []
for r in all_rows:
    gi = gold_index(r)
    if gi is None:
        unresolved.append(r)
    else:
        r["_gold"] = gi
        usable.append(r)

print("rows with a resolvable gold option: {}/{}".format(len(usable), len(all_rows)))
if unresolved:
    print("  DROPPED {} rows whose answer matches no option:".format(len(unresolved)))
    for r in unresolved[:3]:
        print("     [{}] answer={!r} options={}".format(
            r["id"], r["answer"], [str(o)[:18] for o in (r.get("options") or [])]))
assert len(usable) >= N_ROWS, "not enough usable rows"

random.seed(SEED)
sample = random.sample(usable, N_ROWS)

n_opts = Counter(len(r["options"]) for r in sample)
print("\nSampled {} rows | options per row: {}".format(len(sample), dict(n_opts)))
print("  languages     : {}".format(len(Counter(r["language"] for r in sample))))
pos = Counter(r["_gold"] for r in sample)
print("  gold position : {}".format(
    {LETTERS_ALL[k]: v for k, v in sorted(pos.items())}))
top_pos = max(pos.values()) / len(sample)
rand_base = sum(1.0 / len(r["options"]) for r in sample) / len(sample)
print("  most common position holds {:.1%}".format(top_pos))
print("  random guessing would score {:.1%}".format(rand_base))

r = sample[0]
print("\n--- example row ---")
print("  Q: {}".format(" ".join(str(r["question"]).split())[:96]))
for i, o in enumerate(r["options"]):
    mark = " <- gold" if i == r["_gold"] else ""
    print("     {}. {}{}".format(LETTERS_ALL[i], str(o)[:70], mark))

### Cell 5 - Build the prompt

The options are re-labelled `A`, `B`, `C`... regardless of how the file stores
the gold, so the model always sees the same shape and Cell 6 can score by
letter.

`CFG["task"]` from the registry supplies the one task-specific sentence. That
is the only thing that differs between the five datasets - everything else in
the prompt, and all of the scoring, is shared.

**No few-shot examples.** With a constrained single-token answer the format
cannot go wrong, so examples would only add prompt length - and drawing them
from the same file risks leaking a scored row.

In [ ]:
def build_query(row):
    opts = "\n".join("{}. {}".format(LETTERS_ALL[i], str(o).strip())
                      for i, o in enumerate(row["options"]))
    letters = "/".join(LETTERS_ALL[:len(row["options"])])
    return ("{}\n\nQuestion:\n{}\n\nOptions:\n{}\n\n"
            "Answer with one letter ({}):").format(
                CFG["task"], " ".join(str(row["question"]).split()), opts, letters)


SYSTEM_PROMPT = (
    "You are answering multiple-choice questions from an Indian-language "
    "benchmark.\n\n"
    "Reply with a SINGLE letter naming one of the given options. Output only "
    "that letter - no words, no punctuation, no explanation."
)


def build_completion(row):
    return SYSTEM_PROMPT + "\n\n" + build_query(row)


def build_chat_messages(row):
    return [{"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user",   "content": build_query(row)}]


print("=" * 70)
print(build_completion(sample[0]))
print("=" * 70)
print("[gold: {}. {}]".format(LETTERS_ALL[sample[0]["_gold"]],
                              str(sample[0]["options"][sample[0]["_gold"]])[:60]))

### Cell 6 - Constrained answering

`choose()` runs **one forward pass** and takes the argmax over the option
letters' token ids at the final position.

Three consequences:

- The model cannot answer off-list, so there is no output parsing, no retries
  and no unparseable rows. Nothing can be lost to formatting - the failure mode
  that made the comi_lingua POS run unusable is structurally impossible here.
- It is far faster than generating: 200 rows take about a minute per model.
- The number of candidate letters follows each row's own option count, so the
  same code handles 2, 4 and 5 options without special-casing.

`letter_token_ids` collects each letter's first token both bare (`A`) and
space-prefixed (` A`), since tokenizers differ on which one a reply starts with.

In [ ]:
def prompt_style(tokenizer):
    # base checkpoints have no chat template at all
    return "chat" if getattr(tokenizer, "chat_template", None) else "completion"


def letter_token_ids(tokenizer, n):
    ids = {}
    for let in LETTERS_ALL[:n]:
        variants = set()
        for form in (let, " " + let):
            enc = tokenizer.encode(form, add_special_tokens=False)
            if enc:
                variants.add(enc[0])
        ids[let] = sorted(variants)
    return ids


def format_prompt(tokenizer, row):
    if prompt_style(tokenizer) == "completion":
        return build_completion(row)
    msgs = build_chat_messages(row)
    try:
        return tokenizer.apply_chat_template(
            msgs, tokenize=False, add_generation_prompt=True)
    except Exception:
        # some templates (Gemma) reject a system role - fold it into the user
        # turn rather than dropping the instructions
        merged = [{"role": "user",
                   "content": msgs[0]["content"] + "\n\n" + msgs[1]["content"]}]
        return tokenizer.apply_chat_template(
            merged, tokenize=False, add_generation_prompt=True)


@torch.no_grad()
def choose(model, tokenizer, tok_cache, row):
    n = len(row["options"])
    if n not in tok_cache:
        tok_cache[n] = letter_token_ids(tokenizer, n)
    ids = tok_cache[n]

    text   = format_prompt(tokenizer, row)
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    logits = model(**inputs).logits[0, -1]

    best = max(LETTERS_ALL[:n],
               key=lambda let: max(logits[i].item() for i in ids[let]))
    return LETTERS_ALL.index(best)


def clear_hf_cache():
    shutil.rmtree("/root/.cache/huggingface/hub/", ignore_errors=True)
    gc.collect()
    torch.cuda.empty_cache()

print("Answering functions defined")

### Cell 7 - Load-or-cache, and the batched runner

`load_model` implements download-once: if `models/<name>_4bit` exists in Drive
it is loaded directly (already 4-bit, so passing a fresh `BitsAndBytesConfig`
would conflict and is omitted); otherwise the repo is downloaded, quantised and
saved to Drive. `trust_remote_code` stays off - repo-shipped modelling code is
often written against an older transformers API.

Each finished batch is appended to `judge_progress_<dataset>/<model>.jsonl`
before the next begins, so a disconnect costs at most `BATCH_SIZE` rows and a
completed model is skipped without loading its weights.

In [ ]:
def cache_path(spec):
    return os.path.join(CACHE_DIR, spec["name"] + "_4bit")


def load_model(spec):
    cached     = cache_path(spec)
    from_drive = DRIVE_OK and os.path.isfile(os.path.join(cached, "config.json"))
    source     = cached if from_drive else spec["repo"]

    kwargs = {"device_map": "auto", "trust_remote_code": False}
    if from_drive:
        how = "Drive cache (already 4-bit)"
    else:
        kwargs["quantization_config"] = bnb_config
        how = "HuggingFace download -> 4-bit"
    if spec.get("attn"):
        kwargs["attn_implementation"] = spec["attn"]
        how += ", attn=" + spec["attn"]

    print("  loading {} [{}]".format(source, how))
    tokenizer = AutoTokenizer.from_pretrained(source)
    model = AutoModelForCausalLM.from_pretrained(source, **kwargs).eval()

    if not from_drive and DRIVE_OK:
        print("  saving 4-bit copy to {} (one time)...".format(cached))
        os.makedirs(cached, exist_ok=True)
        model.save_pretrained(cached)
        tokenizer.save_pretrained(cached)
        print("  saved - future runs skip the download")

    print("  ready | VRAM: {:.2f}GB | prompt style: {}".format(
        torch.cuda.memory_allocated() / 1e9, prompt_style(tokenizer)))
    return model, tokenizer


def run_model(spec, rows):
    prog = os.path.join(PROG_DIR, spec["name"] + ".jsonl")

    done = {}
    if os.path.exists(prog):
        with open(prog, encoding="utf-8") as f:
            for line in f:
                item = json.loads(line)
                done[item["id"]] = item
        print("  resuming - {}/{} already answered".format(len(done), len(rows)))

    remaining = [r for r in rows if r["id"] not in done]
    if not remaining:
        print("  {} already complete - skipping load".format(spec["name"]))
        return done

    model, tokenizer = load_model(spec)
    tok_cache = {}
    total_batches = (len(remaining) + BATCH_SIZE - 1) // BATCH_SIZE

    for start in range(0, len(remaining), BATCH_SIZE):
        batch, results = remaining[start:start + BATCH_SIZE], []
        for row in batch:
            picked = choose(model, tokenizer, tok_cache, row)
            results.append({"id": row["id"], "picked": picked,
                            "gold": row["_gold"],
                            "correct": int(picked == row["_gold"])})
        with open(prog, "a", encoding="utf-8") as f:
            for item in results:
                f.write(json.dumps(item, ensure_ascii=False) + "\n")
        done.update({i["id"]: i for i in results})
        acc = sum(v["correct"] for v in done.values()) / len(done)
        print("  batch {}/{} saved - {}/{} rows | running acc {:.1%}".format(
            start // BATCH_SIZE + 1, total_batches, len(done), len(rows), acc))

    del model, tokenizer
    clear_hf_cache()
    print("  {} complete".format(spec["name"]))
    return done

print("Runner defined")

### Cell 8 - Run all three models

One at a time - loaded, scored, unloaded - so peak VRAM stays near 6 GB rather
than the ~17 GB all three would need together.

Fast, because each row is a single forward pass: budget a couple of minutes per
model plus downloads on the first run. Safe to re-run - anything already scored
is skipped.

In [ ]:
answers = {}
for spec in MODELS:
    print("\n=== {} ===".format(spec["name"]))
    answers[spec["name"]] = run_model(spec, sample)

print("\nAll models done")

### Cell 9 - Assign difficulty and write the schema

Each model contributes 1 if its pick matches gold. The three votes sum into
Easy / Medium / Hard - no threshold, because an MCQ answer is already binary.

Output rows are rebuilt key-by-key from `SCHEMA_KEYS`, so all 14 fields are
preserved in schema order and the internal `_gold` field never leaks. Questions,
options and answers pass through untouched.

In [ ]:
def get_difficulty(votes):
    score = sum(votes)
    if score == 3:
        return "Easy"
    elif score == 2:
        return "Medium"
    else:
        return "Hard"


final_results, audit = [], []

for row in sample:
    votes = [answers[s["name"]][row["id"]]["correct"] for s in MODELS]
    difficulty = get_difficulty(votes)

    enriched = {**row, "difficulty": difficulty}
    if SET_EVAL_METRIC:
        enriched["eval_metric"] = SET_EVAL_METRIC
    final_results.append({k: enriched.get(k) for k in SCHEMA_KEYS})

    audit.append({
        "id":         row["id"],
        "dataset":    DATASET,
        "difficulty": difficulty,
        "votes":      votes,
        "gold":       LETTERS_ALL[row["_gold"]],
        "picks":      {s["name"]: LETTERS_ALL[answers[s["name"]][row["id"]]["picked"]]
                       for s in MODELS},
        "language":   row["language"],
        "question":   " ".join(str(row["question"]).split()),
        "options":    [str(o) for o in row["options"]],
    })

with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
    for item in final_results:
        f.write(json.dumps(item, ensure_ascii=False) + "\n")
with open(AUDIT_FILE, "w", encoding="utf-8") as f:
    for item in audit:
        f.write(json.dumps(item, ensure_ascii=False) + "\n")

print("Saved -> {} ({} rows)".format(OUTPUT_FILE, len(final_results)))
print("Audit -> {}".format(AUDIT_FILE))

### Cell 10 - Verify and report

1. **Schema** - all 14 keys in order, no nulls in `difficulty`, no `_gold` leak,
   and questions/options/answers identical to the input.
2. **Difficulty distribution.**
3. **Per-model accuracy against two baselines** - random guessing, and always
   picking the most common gold position. A model below random is broken, not
   weak; a model near the position baseline may be answering by position rather
   than by reading.
4. **Pick distribution per model.** If a model's picks pile onto one letter, it
   has a positional bias and its votes are close to noise - the same degenerate
   pattern that made an earlier run unusable.
5. **Accuracy by language**, since these files span 11 Indian languages and a
   model may be competent in some and not others.

In [ ]:
bad_keys = [r["id"] for r in final_results if list(r.keys()) != SCHEMA_KEYS]
missing  = [r["id"] for r in final_results if r["difficulty"] is None]
leaked   = [r["id"] for r in final_results if any(k.startswith("_") for k in r)]
src_by_id = {r["id"]: r for r in all_rows}
altered = [r["id"] for r in final_results
           if r["question"] != src_by_id[r["id"]]["question"]
           or r["answer"] != src_by_id[r["id"]]["answer"]
           or r["options"] != src_by_id[r["id"]]["options"]]
print("Schema check : {} rows | wrong keys: {} | null difficulty: {} | _gold leaked: {}".format(
    len(final_results), len(bad_keys), len(missing), len(leaked)))
print("input fields altered: {}".format(len(altered)))

dist  = Counter(r["difficulty"] for r in final_results)
total = len(final_results)
print("\nDifficulty distribution ({}):".format(DATASET))
for level in ["Easy", "Medium", "Hard"]:
    n = dist.get(level, 0)
    print("  {:<7}: {:4d}  ({:.1f}%)".format(level, n, n / total * 100))

rand_base = sum(1.0 / len(r["options"]) for r in sample) / total
pos = Counter(r["_gold"] for r in sample)
pos_base = max(pos.values()) / total
print("\nAccuracy:")
for s in MODELS:
    acc = sum(v["correct"] for v in answers[s["name"]].values()) / total
    flag = "  <- at/below random" if acc <= rand_base else ""
    print("  {:<10} {:.1%}{}".format(s["name"], acc, flag))
print("  {:<10} {:.1%}  <- random guessing".format("random", rand_base))
print("  {:<10} {:.1%}  <- always pick the most common gold position".format(
    "position", pos_base))

print("\nPick distribution (positional bias check):")
for s in MODELS:
    c = Counter(LETTERS_ALL[v["picked"]] for v in answers[s["name"]].values())
    top = max(c.values()) / total
    flag = "  <- one letter dominates, votes are near noise" if top > 0.7 else ""
    print("  {:<10} {}{}".format(s["name"], dict(sorted(c.items())), flag))

print("\nAccuracy by language:")
langs = Counter(r["language"] for r in sample)
for lg, n in langs.most_common(12):
    ids = [r["id"] for r in sample if r["language"] == lg]
    line = "  {:<10} n={:3d}".format(lg, n)
    for s in MODELS:
        a = sum(answers[s["name"]][i]["correct"] for i in ids) / n
        line += "  {}={:.0%}".format(s["name"][:4], a)
    print(line)

print("\n--- 2 sample rows ---")
for a in audit[:2]:
    print("\n  {} [{}] gold={} picks={}".format(
        a["id"], a["difficulty"], a["gold"], a["picks"]))
    print("    Q: {}".format(a["question"][:88]))